In [ ]:
from bs4 import BeautifulSoup

In [ ]:
from bs4 import BeautifulSoup

html_content = '''
<div class="outer-cell mdl-cell mdl-cell--12-col mdl-shadow--2dp">
    <div class="mdl-grid">
        <div class="header-cell mdl-cell mdl-cell--12-col">
            <p class="mdl-typography--title">Google Pay<br></p>
        </div>
        <div class="content-cell mdl-cell mdl-cell--6-col mdl-typography--body-1">
            Paid ₹24.00 to Example Merchant using Bank Account XXXXXX0000<br>
            14 Feb 2024, 21:01:43 IST
        </div>
        <div class="content-cell mdl-cell mdl-cell--6-col mdl-typography--body-1 mdl-typography--text-right"></div>
        <div class="content-cell mdl-cell mdl-cell--12-col mdl-typography--caption">
            <b>Products:</b><br>
            Google Pay<br>
            <b>Details:</b><br>
            +xv4mzyuSuOU+BGd<br>
            Completed<br>
        </div>
    </div>
</div>
'''

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Extracting data
header = soup.find('p', class_='mdl-typography--title').text.strip()
transaction_info = soup.find(class_='content-cell mdl-cell mdl-cell--6-col mdl-typography--body-1').text.strip()

# Find all <b> tags
b_tags = soup.find_all('b')

# Check if there are any <b> tags found
if b_tags and len(b_tags) >= 2:
    products = b_tags[0].find_next_sibling(text=True).strip()
    details = b_tags[1].find_next_sibling(text=True).strip()
else:
    products = None
    details = None

print("Header:", header)
print("Transaction Info:", transaction_info)
print("Products:", products)
print("Details:", details)


In [ ]:
from bs4 import BeautifulSoup
from openpyxl import Workbook
import re

# Read the HTML file
with open('/content/output.html') as file:
    html_content = file.read()

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Create a new Excel workbook
workbook = Workbook()
sheet = workbook.active

# Set the header row
header_row = ['Amount', 'Recipient', 'Transaction Type', 'Bank Account', 'Date', 'Time']
sheet.append(header_row)

# Find all elements with class "outer-cell"
elements = soup.find_all(class_='outer-cell')

# Iterate over each element and extract information
for element in elements:
    transaction_info = element.find(class_='content-cell mdl-cell mdl-cell--6-col mdl-typography--body-1').text.strip()
    print("Transaction Info:", transaction_info)

    # Extracting different parts of transaction_info
    amount_match = re.search(r'₹([\d.]+)', transaction_info)
    if amount_match:
        amount = amount_match.group(1)
    else:
        amount = ""

    parts = transaction_info.split('using Bank Account')
    amount_location = parts[0].strip()

    if len(parts) > 1:
        # Extract sender_bank_account, date_time
        match = re.search(r'(\d{2} [A-Za-z]+ \d{4}, \d{2}:\d{2}:\d{2} IST)', parts[1])
        if match:
            date_time = match.group(1)
            sender_bank_account = parts[1].replace(date_time, '').strip()
        else:
            date_time = ""
            sender_bank_account = parts[1].strip()

        print("Sender Bank Account:", sender_bank_account)
        print("Date Time:", date_time)

        # Extract recipient's name
        recipient_match = re.search(r'to\s+([\w\s]+)', sender_bank_account)
        if recipient_match:
            recipient = recipient_match.group(1).strip()
        else:
            recipient = ""

        # Extracting sender, bank account, date, and time
        sender_split = sender_bank_account.split('to')
        if len(sender_split) > 1:
            sender = sender_split[1].strip()
            bank_account = 'XXXXXX' + sender_split[0].strip()[-4:]
        else:
            sender = ""
            bank_account = ""

        date_time = date_time.split(',')
        date = date_time[0].strip()
        time = date_time[1].strip()

    else:
        sender = ""
        bank_account = ""
        date = ""
        time = ""
        recipient = ""

    # Append the extracted data to the Excel sheet
    row = [amount, recipient, 'Paid', bank_account, date, time]
    sheet.append(row)

# Save the workbook to a file
workbook.save(filename='scraped_data.xlsx')
